In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

In [30]:
# SPARK SESSION
spark = SparkSession.builder \
    .appName("Analisis_Clientes_Behavioural") \
    .getOrCreate()

In [31]:
df_beh = spark.read.parquet("/home/jovyan/work/data/BEHAVIOURAL_CLEAN")
df_cli = spark.read.parquet("/home/jovyan/work/data/CLIENTS_CLEAN")

In [32]:
from pyspark.sql import functions as F

# 1. CLIENT_ID ÚNICOS en cada dataset
ids_beh = df_beh.select("CLIENT_ID").distinct()
ids_cli = df_cli.select("CLIENT_ID").distinct()

# 2. CLIENTES COMUNES (IDs únicos)
ids_comunes = ids_beh.intersect(ids_cli)

num_comunes = ids_comunes.count()
print(f"CLIENT_ID comunes (únicos) entre df_beh y df_cli: {num_comunes}")

CLIENT_ID comunes (únicos) entre df_beh y df_cli: 45668


In [33]:
# 3. Contar cuántas veces aparecen esos ID comunes dentro de df_beh
reps_beh = df_beh.join(ids_comunes, "CLIENT_ID", "inner").count()

# 4. Contar cuántas veces aparecen esos ID comunes dentro de df_cli
reps_cli = df_cli.join(ids_comunes, "CLIENT_ID", "inner").count()

# 5. Total de repeticiones entre ambos datasets
total_repeticiones = reps_beh + reps_cli

print(f"Estos CLIENT_ID comunes aparecen {total_repeticiones} veces en total entre df_beh y df_cli.")
print(f"- En df_beh: {reps_beh} filas")
print(f"- En df_cli: {reps_cli} filas")

Estos CLIENT_ID comunes aparecen 1734602 veces en total entre df_beh y df_cli.
- En df_beh: 1688934 filas
- En df_cli: 45668 filas


In [34]:
# IDs únicos de beh que NO están en cli
ids_solo_beh = (
    df_beh.select("CLIENT_ID").distinct()
          .join(df_cli.select("CLIENT_ID").distinct(), "CLIENT_ID", "left_anti")
)

# Traer TODAS las columnas de df_beh de esos IDs únicos
# (sin dropDuplicates)
solo_beh = df_beh.join(ids_solo_beh, "CLIENT_ID", "inner")

solo_beh_count = solo_beh.count()

print("CLIENT_ID únicos solo en df_beh:", solo_beh_count)
solo_beh.show(solo_beh_count, truncate=False)


# IDs únicos de cli que NO están en beh
ids_solo_cli = (
    df_cli.select("CLIENT_ID").distinct()
          .join(df_beh.select("CLIENT_ID").distinct(), "CLIENT_ID", "left_anti")
)

# Traer TODAS las columnas de df_cli de esos IDs únicos
# (sin dropDuplicates)
solo_cli = df_cli.join(ids_solo_cli, "CLIENT_ID", "inner")

solo_cli_count = solo_cli.count()

print("CLIENT_ID únicos solo en df_cli:", solo_cli_count)
solo_cli.show(solo_cli_count, truncate=False)

CLIENT_ID únicos solo en df_beh: 35920


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



CLIENT_ID únicos solo en df_cli: 108539


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [35]:
solo_beh.toPandas().to_csv("/home/jovyan/work/data/solo_beh.csv", index=False)
solo_cli.toPandas().to_csv("/home/jovyan/work/data/solo_cli.csv", index=False)

In [36]:
solo_beh

DataFrame[CLIENT_ID: string, CONTRACT_ID: string, DATE: date, CREDICT_CARD_BALANCE: double, CREDIT_CARD_LIMIT: double, CREDIT_CARD_DRAWINGS_ATM: double, CREDIT_CARD_DRAWINGS: double, CREDIT_CARD_DRAWINGS_POS: double, CREDIT_CARD_DRAWINGS_OTHER: double, CREDIT_CARD_PAYMENT: double, NUMBER_DRAWINGS_ATM: double, NUMBER_DRAWINGS: int, NUMBER_INSTALMENTS: double]

In [37]:
solo_cli

DataFrame[CLIENT_ID: string, NON_COMPLIANT_CONTRACT: int, NAME_PRODUCT_TYPE: string, GENDER: string, TOTAL_INCOME: double, AMOUNT_PRODUCT: double, INSTALLMENT: double, EDUCATION: string, MARITAL_STATUS: string, HOME_SITUATION: string, REGION_SCORE: double, AGE_IN_YEARS: double, JOB_SENIORITY: double, HOME_SENIORITY: double, LAST_UPDATE: double, OWN_INSURANCE_CAR: string, FAMILY_SIZE: double, PROACTIVE_SCORING: double, BEHAVIORAL_SCORING: double, DAYS_LAST_INFO_CHANGE: double, NUMBER_OF_PRODUCTS: double, OCCUPATION: string, DIGITAL_CLIENT: int, HOME_OWNER: string, EMPLOYER_ORGANIZATION_TYPE: string, NUM_PREVIOUS_LOAN_APP: double, LOAN_ANNUITY_PAYMENT_MAX: double, LOAN_ANNUITY_PAYMENT_MIN: double, LOAN_ANNUITY_PAYMENT_SUM: double, LOAN_APPLICATION_AMOUNT_MAX: double, LOAN_APPLICATION_AMOUNT_MIN: double, LOAN_APPLICATION_AMOUNT_SUM: double, LOAN_CREDIT_GRANTED_MAX: double, LOAN_CREDIT_GRANTED_MIN: double, LOAN_CREDIT_GRANTED_SUM: double, LOAN_VARIABLE_RATE_MAX: double, LOAN_VARIABLE_RATE_